# M4 — Per-Token Confidence Routing

For each token in the teacher's response:
- **High confidence** (teacher logprob > τ) → standard CE (mimic teacher)
- **Low confidence** (teacher logprob ≤ τ) → entropy matching (match uncertainty)

Plus auxiliary `L_dec_only` to preserve reasoning pressure.

τ = median teacher logprob across training set.

## Technical notes
- Memory-efficient: entropy computed only on low-confidence token positions
- Precomputed dataset (DatasetFast pattern)
- One model per kernel, restart between models
- bf16 LoRA, no quantization

In [3]:
# Cell 0: Config
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

import torch
torch.cuda.set_per_process_memory_fraction(0.70)

TRAIN_MODEL = "qwen25_7b_base"

print(f"Will train: {TRAIN_MODEL}")

Will train: qwen25_7b_base


In [4]:
# Cell 1: Imports
import os, sys, json, math, re
from typing import List, Dict, Any, Tuple
from tqdm import tqdm

import torch
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

sys.path.insert(0, os.path.expanduser("~/kd_project"))
from shared_utils import (
    RUNS_DIR, MAX_SEQ_LEN, EPOCHS, LR, SEED, LAMBDA,
    STUDENTS, load_data, load_student,
    find_decision_span_chars, entropy_from_logprobs,
    build_prompt_text, tokenize_and_mask, FlexibleCollator,
)

OUT_DIR = os.path.join(RUNS_DIR, "m4_token_confidence_routing")
os.makedirs(OUT_DIR, exist_ok=True)

ROUTE_TEMP = 0.5    # sigmoid sharpness
W_DEC_AUX = 0.2     # auxiliary L_dec_only weight
LAMBDA_ENT = 0.1    # entropy matching weight for uncertain tokens

prompts, teacher_rows = load_data()

# Compute TAU = median teacher logprob
all_lps = []
for r in teacher_rows:
    for st in (r.get("logprobs_steps") or []):
        lp = (st.get("chosen") or {}).get("logprob")
        if lp is not None: all_lps.append(float(lp))
all_lps.sort()
TAU = all_lps[len(all_lps) // 2] if all_lps else -0.5
print(f"TAU (median teacher logprob): {TAU:.3f} (n={len(all_lps)})")

✅ Loaded 5000 teacher samples from: data\clinical_pharm_teacher_train_6000.jsonl
TAU (median teacher logprob): -0.001 (n=1003041)


In [5]:
# Cell 2: Token alignment helpers
def build_token_teacher_info(row):
    steps = row.get("logprobs_steps") or []
    info, buf = [], ""
    for st in steps:
        ch = st.get("chosen", {})
        tok_str = ch.get("token", "")
        lp = ch.get("logprob")
        topk = st.get("topk", [])
        topk_lps = [t.get("logprob") for t in topk if t.get("logprob") is not None]
        start = len(buf); buf += tok_str; end = len(buf)
        info.append({"logprob": float(lp) if lp is not None else None,
                      "topk_logprobs": topk_lps, "cs": start, "ce": end})
    return info

def align_teacher_to_student(teacher_info, offsets, answer_start):
    n = len(offsets)
    t_lp = [None] * n
    t_topk = [[] for _ in range(n)]
    if not teacher_info: return t_lp, t_topk
    for i, (s, e) in enumerate(offsets):
        if e <= s: continue
        ans_s, ans_e = s - answer_start, e - answer_start
        if ans_s < 0: continue
        best_j, best_ol, overlapping_lps = -1, 0, []
        for j, ti in enumerate(teacher_info):
            ol_s, ol_e = max(ans_s, ti["cs"]), min(ans_e, ti["ce"])
            if ol_e > ol_s:
                if ti["logprob"] is not None: overlapping_lps.append(ti["logprob"])
                if ol_e - ol_s > best_ol: best_ol = ol_e - ol_s; best_j = j
        if overlapping_lps: t_lp[i] = sum(overlapping_lps) / len(overlapping_lps)
        if best_j >= 0: t_topk[i] = teacher_info[best_j]["topk_logprobs"]
    return t_lp, t_topk

print("✅ Alignment helpers ready")

✅ Alignment helpers ready


In [6]:
# Cell 3: Precomputed Dataset — OPTIMIZED alignment
class M4DatasetFast(Dataset):
    def __init__(self, rows, prompts, tokenizer, is_instruct, tau):
        print("  Precomputing dataset...")
        self.items = []
        skipped = 0
        for idx in tqdm(range(len(rows)), desc="  Tokenizing"):
            r = rows[idx]
            prompt = prompts[r["id"]]
            answer = r["teacher_text"]
            prompt_text = build_prompt_text(tokenizer, prompt, is_instruct)
            full_text = prompt_text + answer
            answer_start = len(prompt_text)
            enc = tokenizer(full_text, truncation=True, max_length=MAX_SEQ_LEN, return_offsets_mapping=True)
            input_ids, offsets = enc["input_ids"], enc["offset_mapping"]

            labels = [-100] * len(input_ids)
            for i, (s, e) in enumerate(offsets):
                if e > s and s >= answer_start: labels[i] = input_ids[i]

            # Build teacher char→logprob lookup (fast)
            steps = r.get("logprobs_steps") or []
            teacher_lp_by_char = {}  # char_pos → logprob
            teacher_topk_by_char = {}  # char_pos → [logprobs]
            buf = ""
            for st in steps:
                ch = st.get("chosen", {})
                tok_str = ch.get("token", "")
                lp = ch.get("logprob")
                topk = st.get("topk", [])
                topk_lps = [t.get("logprob") for t in topk if t.get("logprob") is not None]
                start_c = len(buf)
                buf += tok_str
                if lp is not None:
                    teacher_lp_by_char[start_c] = float(lp)
                if topk_lps:
                    teacher_topk_by_char[start_c] = topk_lps

            # Fast alignment: for each student token, find teacher char
            conf_mask = torch.zeros(len(input_ids), dtype=torch.float)
            teacher_ent = torch.zeros(len(input_ids), dtype=torch.float)

            # Pre-sort teacher positions for binary search
            teacher_positions = sorted(teacher_lp_by_char.keys())
            topk_positions = sorted(teacher_topk_by_char.keys())

            for i, (s, e) in enumerate(offsets):
                if labels[i] == -100: continue
                ans_s = s - answer_start
                if ans_s < 0: continue

                # Find closest teacher token (by char start)
                # Simple: find teacher position that overlaps [ans_s, ans_e)
                ans_e = e - answer_start
                best_lp = None
                for tp in teacher_positions:
                    if tp >= ans_e: break
                    if tp >= ans_s:
                        best_lp = teacher_lp_by_char[tp]
                        break

                if best_lp is not None:
                    x = (best_lp - tau) / ROUTE_TEMP
                    x = max(-20.0, min(20.0, x))
                    conf_mask[i] = 1.0 / (1.0 + math.exp(-x))

                # Teacher entropy from topk
                best_topk = None
                for tp in topk_positions:
                    if tp >= ans_e: break
                    if tp >= ans_s:
                        best_topk = teacher_topk_by_char[tp]
                        break

                if best_topk:
                    clean = [x for x in best_topk if x is not None]
                    if clean:
                        lps = torch.tensor(clean, dtype=torch.float32)
                        teacher_ent[i] = entropy_from_logprobs(lps).item()
            # Decision mask
            dec_mask = torch.zeros(len(input_ids), dtype=torch.bool)
            dec_span = find_decision_span_chars(answer)
            if dec_span:
                ds, de = dec_span
                for i, (s, e) in enumerate(offsets):
                    if labels[i] != -100 and not (e <= answer_start+ds or s >= answer_start+de):
                        dec_mask[i] = True

            self.items.append({
                "input_ids": torch.tensor(input_ids, dtype=torch.long),
                "attention_mask": torch.ones(len(input_ids), dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
                "confidence_mask": conf_mask,
                "teacher_ent": teacher_ent,
                "dec_mask": dec_mask,
            })

        print(f"  ✅ Precomputed {len(self.items)} samples")

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print("✅ Dataset class ready")

✅ Dataset class ready


In [ ]:
# Cell 4: Trainer — memory-efficient routing
class M4Trainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._ce_loss = torch.nn.CrossEntropyLoss(reduction="none")

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        conf = inputs.pop("confidence_mask")
        t_ent = inputs.pop("teacher_ent")
        dm = inputs.pop("dec_mask")

        out = model(**inputs)
        logits = out.logits
        sl = logits[:, :-1, :].contiguous()
        ll = labels[:, 1:].contiguous()
        cf = conf[:, 1:]
        te = t_ent[:, 1:]
        d = dm[:, 1:]
        vocab, B = sl.size(-1), sl.size(0)

        tl = self._ce_loss(sl.view(-1, vocab), ll.view(-1)).view(ll.size())
        active = (ll != -100).float()

        # CE on high-confidence tokens (weighted by conf)
        ce_comp = tl * cf * active

        # Entropy matching on low-confidence tokens — limit to top 20 per sample
        low_conf_mask = ((1.0 - cf) > 0.5) & (ll != -100)
        ent_diff = torch.zeros(tl.shape, dtype=torch.float32, device=tl.device)
        for b in range(B):
            idx = low_conf_mask[b].nonzero(as_tuple=True)[0]
            if len(idx) == 0: continue
            if len(idx) > 20:  # cap at 20 most uncertain tokens
                conf_vals = cf[b, idx]
                _, topk_idx = conf_vals.topk(20, largest=False)
                idx = idx[topk_idx]
            local_logits = sl[b, idx, :]
            local_probs = torch.softmax(local_logits, dim=-1)
            local_ent = -(local_probs * torch.log(local_probs + 1e-9)).sum(-1)
            ent_diff[b, idx] = (local_ent - te[b, idx].to(sl.device)) ** 2

        total = active.sum().clamp(min=1.0)
        L_routed = ce_comp.sum() / total + LAMBDA_ENT * (ent_diff * active).sum() / total

        # Auxiliary L_dec_only
        da = d.float() * active
        L_dec = (tl * da).sum() / da.sum().clamp(min=1.0)

        loss = L_routed + W_DEC_AUX * L_dec
        return (loss, out) if return_outputs else loss

print("✅ Trainer ready")

✅ Trainer ready


: 

In [ ]:
# Cell 5: Train with checkpointing + resume
cfg = STUDENTS[TRAIN_MODEL]
print(f"\n{'='*50} M4: {TRAIN_MODEL} {'='*50}")
out_path = os.path.join(OUT_DIR, TRAIN_MODEL)
os.makedirs(out_path, exist_ok=True)

tokenizer, model = load_student(TRAIN_MODEL, cfg)

if "7b" in TRAIN_MODEL:
    model.gradient_checkpointing_enable()

dataset = M4DatasetFast(teacher_rows, prompts, tokenizer, cfg["is_instruct"], TAU)
collator = FlexibleCollator(tokenizer, extra_1d_fields=["confidence_mask", "teacher_ent", "dec_mask"])

if "1p5b" in TRAIN_MODEL:
    bs, ga = 4, 8
elif "3b" in TRAIN_MODEL:
    bs, ga = 2, 16
else:
    bs, ga = 1, 32

use_bf16 = "7b" not in TRAIN_MODEL  # 8-bit can't use bf16

trainer = M4Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=out_path, num_train_epochs=EPOCHS,
        per_device_train_batch_size=bs, gradient_accumulation_steps=ga,
        learning_rate=LR, logging_steps=25,
        save_strategy="steps", save_steps=70, save_total_limit=2,
        bf16=use_bf16, seed=SEED, report_to="none",
        remove_unused_columns=False, dataloader_num_workers=0,
        gradient_checkpointing=True,
    ),
    train_dataset=dataset, data_collator=collator,
)

import glob
ckpts = sorted(glob.glob(os.path.join(out_path, "checkpoint-*")))
if ckpts:
    print(f"  Resuming from {ckpts[-1]}")
    trainer.train(resume_from_checkpoint=ckpts[-1])
else:
    trainer.train()

model.save_pretrained(out_path)
tokenizer.save_pretrained(out_path)
print(f"\n✅ {TRAIN_MODEL} saved → {out_path}")


================================================== M4: qwen25_7b_base ==================================================
  Loading qwen25_7b_base from Qwen/Qwen2.5-7B...


  Using 4-bit QLoRA quantization (Windows WDDM stability)


c:\Users\adishalit1\AppData\Local\anaconda3\envs\kd\Lib\site-packages\accelerate\utils\modeling.py:804: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  _ = torch.tensor([0], device=i)
Loading weights: 100%|██████████| 339/339 [00:10<00:00, 32.89it/s] 


  ✅ qwen25_7b_base: 10,092,544 trainable / 4,363,064,832 total params
  Precomputing dataset...


  Tokenizing: 100%|██████████| 5000/5000 [00:30<00:00, 165.19it/s]


  ✅ Precomputed 5000 samples
  Resuming from runs\m4_token_confidence_routing\qwen25_7b_base\checkpoint-70


Step,Training Loss
75,13.508620
100,13.008761


: 